# PubMedBERT ADR Classifier

Fine-tune `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext` on the exact existing ADE train/validation/test CSV splits. The test set is evaluated only after validation-based model selection. This notebook does not claim clinical validation.

In [1]:
import sys

print(f"Python interpreter: {sys.executable}")

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Python interpreter: c:\Users\ayush\Downloads\TB-ADR\TB-ADR\.venv\Scripts\python.exe
PyTorch version: 2.14.1+cu126
CUDA available: True


In [2]:
from pathlib import Path
import sys, json, random
import numpy as np

import pandas as pd
import torch

workspace = Path.cwd().parent
sys.path.insert(0, str(workspace))
from src.nlp.transformer import seed_everything, classification_metrics, save_json

SEED = 42
MODEL_NAME = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext'
MAX_LENGTH = 256
BATCH_SIZE = 16 if torch.cuda.is_available() else 4
EPOCHS = 3
VALIDATION_PATIENCE = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seed_everything(SEED)
data_dir = workspace / 'data' / 'processed' / 'ade_corpus'
train = pd.read_csv(data_dir / 'train.csv')
validation = pd.read_csv(data_dir / 'validation.csv')
test = pd.read_csv(data_dir / 'test.csv')
print({'device': str(DEVICE), 'cuda_available': torch.cuda.is_available(), 'max_length': MAX_LENGTH, 'batch_size': BATCH_SIZE, 'train': len(train), 'validation': len(validation), 'test': len(test)})

{'device': 'cuda', 'cuda_available': True, 'max_length': 256, 'batch_size': 16, 'train': 13581, 'validation': 3135, 'test': 4179}


## Sequence length and class weighting

The EDA showed short clinical sentences with a maximum below 1,000 characters. A 256-token cap is a deliberate starting point: it covers typical records while avoiding an automatic 512-token setting. Class-weighted cross entropy addresses imbalance without applying SMOTE to text.

In [3]:
RUN_TRAINING = torch.cuda.is_available()
transformer_status = {'model_name': MODEL_NAME, 'device': str(DEVICE), 'max_length': MAX_LENGTH, 'status': 'not_started'}
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
    from torch.utils.data import Dataset, DataLoader
    from sklearn.metrics import confusion_matrix
except Exception as exc:
    RUN_TRAINING = False
    transformer_status.update({'status': 'unavailable', 'reason': repr(exc)})

if RUN_TRAINING:
    class TextDataset(Dataset):
        def __init__(self, frame):
            self.texts = frame.text.tolist()
            self.labels = frame.label.astype(int).tolist()
        def __len__(self): return len(self.labels)
        def __getitem__(self, index): return {'text': self.texts[index], 'labels': self.labels[index]}

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors='pt')
    def collate(batch):
        encoded = tokenizer([x['text'] for x in batch], truncation=True, max_length=MAX_LENGTH)
        encoded = collator(encoded)
        encoded['labels'] = torch.tensor([x['labels'] for x in batch])
        return encoded
    train_loader = DataLoader(TextDataset(train), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(TextDataset(validation), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    class_counts = torch.bincount(torch.tensor(train.label.tolist()), minlength=2).float()
    weights = (class_counts.sum() / (2 * class_counts)).to(DEVICE)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    best_f1, best_state, patience = -1.0, None, 0
    def evaluate(loader):
        model.eval(); labels=[]; predictions=[]; scores=[]
        with torch.no_grad():
            for batch in loader:
                batch = {k:v.to(DEVICE) for k,v in batch.items()}
                logits = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
                labels.extend(batch['labels'].cpu().tolist()); predictions.extend(logits.argmax(-1).cpu().tolist()); scores.extend(torch.softmax(logits, -1)[:,1].cpu().tolist())
        return labels, predictions, scores
    for epoch in range(EPOCHS):
        model.train()
        for batch in train_loader:
            batch = {k:v.to(DEVICE) for k,v in batch.items()}
            optimizer.zero_grad()
            logits = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
            loss_fn(logits, batch['labels']).backward()
            optimizer.step()
        y, pred, score = evaluate(val_loader)
        metrics = classification_metrics(y, pred, score)
        print({'epoch': epoch + 1, 'validation_adr_f1': metrics['adr_f1'], 'validation_adr_recall': metrics['adr_recall']})
        if metrics['adr_f1'] > best_f1:
            best_f1 = metrics['adr_f1']; best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; patience = 0
        else:
            patience += 1
            if patience >= VALIDATION_PATIENCE: break
    model.load_state_dict(best_state)
    transformer_status.update({'status': 'trained', 'best_validation_adr_f1': best_f1, 'epochs_completed': epoch + 1})
else:
    print('PubMedBERT training not run: CUDA/model resources are not available. No transformer metrics will be fabricated.')

c:\Users\ayush\Downloads\TB-ADR\TB-ADR\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 31186.16it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.se

{'epoch': 1, 'validation_adr_f1': 0.8723864455659697, 'validation_adr_recall': 0.9438377535101404}
{'epoch': 2, 'validation_adr_f1': 0.8913690476190477, 'validation_adr_recall': 0.9344773790951638}
{'epoch': 3, 'validation_adr_f1': 0.8553546592489569, 'validation_adr_recall': 0.9594383775351014}


In [4]:
if not RUN_TRAINING:
    transformer_status.update({'status': 'not_run', 'reason': 'CUDA is unavailable in the current environment; model fine-tuning was not attempted.'})

## Test evaluation and error analysis

In [5]:
model_dir = workspace / 'models' / 'adr_classifier_transformer'
evaluation_dir = workspace / 'evaluation'
if transformer_status['status'] == 'trained':
    test_loader = DataLoader(TextDataset(test), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    y_test, pred_test, score_test = evaluate(test_loader)
    test_metrics = classification_metrics(y_test, pred_test, score_test)
    errors = test[['text','label']].copy(); errors['prediction'] = pred_test; errors['score'] = score_test; errors['error_type'] = 'correct'
    errors.loc[(errors.label == 0) & (errors.prediction == 1), 'error_type'] = 'false_positive'
    errors.loc[(errors.label == 1) & (errors.prediction == 0), 'error_type'] = 'false_negative'
    model_dir.mkdir(parents=True, exist_ok=True); evaluation_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(model_dir); tokenizer.save_pretrained(model_dir)
    errors.to_csv(evaluation_dir / 'transformer_test_errors.csv', index=False)
    save_json({'model': MODEL_NAME, 'config': transformer_status, 'test_metrics': test_metrics}, evaluation_dir / 'transformer_results.json')
    display(pd.DataFrame([test_metrics])); display(errors.error_type.value_counts().rename('count').to_frame())
    print('False positives'); display(errors[errors.error_type == 'false_positive'][['text','label','score']].head(10))
    print('False negatives'); display(errors[errors.error_type == 'false_negative'][['text','label','score']].head(10))
else:
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    save_json({'model': MODEL_NAME, 'config': transformer_status, 'test_metrics': None, 'note': 'Training did not run; no metrics are available.'}, evaluation_dir / 'transformer_results.json')
    print(transformer_status)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.08it/s]


,accuracy,precision_adr,adr_recall,adr_f1,macro_f1,roc_auc,pr_auc,confusion_matrix
0,0.945441,0.835837,0.912178,0.87234,0.918824,0.983059,0.937287,"[[3172, 153], [75, 779]]"


,count
error_type,
correct,3951
false_positive,153
false_negative,75


False positives


,text,label,score
13,Gentamicin alone may be nephrotoxic and should...,0,0.750830
15,"A low frequency, high amplitude rhythmic fluct...",0,0.558092
121,Weight loss during therapy with olanzapine ora...,0,0.985106
123,Monitoring blood concentrations of tacrolimus ...,0,0.876759
147,Temozolomide is a relatively new chemotherapeu...,0,0.984893
151,"Ten months after discontinuing tamoxifen, she ...",0,0.635834
176,Eight deaths that occurred during Upjohn-spons...,0,0.634921
185,The hepatotoxicity is probably due to (-)-epig...,0,0.914289
213,A 32-year-old male schizophrenic patient on zi...,0,0.755633
217,Ovarian torsion: a complication of GIFT.,0,0.511787


False negatives


,text,label,score
2,A few recent individual case reports have sugg...,1,0.479674
33,Clinicians should be cognizant of this possibi...,1,0.175944
63,This case report describes an adolescent with ...,1,0.018964
243,The purpose of this report is to document a ne...,1,0.259362
274,"Therefore, clinicians should pay close attenti...",1,0.427320
431,Proliferation of abnormal bone marrow histiocy...,1,0.497316
506,"In association with this tocolysis, urinary ca...",1,0.320782
532,"Three months following splenectomy, multiple a...",1,0.114184
628,RESULTS: Five of 70 patients who had begun lef...,1,0.108160
710,An unusual cause of burn injury: unsupervised ...,1,0.191915


## Comparison and next stage

The existing classical metrics are loaded for comparison; PubMedBERT values remain unavailable unless training completes. Primary comparison metrics are ADR recall, ADR F1, macro F1, and PR-AUC.

In [6]:
baseline = pd.read_csv(workspace / 'evaluation' / 'adr_baseline_comparison.csv')
comparison = baseline[baseline.evaluation_split == 'test'][['model','accuracy','recall_adr','f1_adr','macro_f1','pr_auc']].copy()
if transformer_status['status'] == 'trained': comparison = pd.concat([comparison, pd.DataFrame([{'model':'PubMedBERT','accuracy':test_metrics['accuracy'],'recall_adr':test_metrics['adr_recall'],'f1_adr':test_metrics['adr_f1'],'macro_f1':test_metrics['macro_f1'],'pr_auc':test_metrics['pr_auc']}])], ignore_index=True)
display(comparison)
print('Next stage: use the saved PubMedBERT model only after reviewing resource availability and comparing it fairly with the fixed classical test results. No clinical validation is implied.')

,model,accuracy,recall_adr,f1_adr,macro_f1,pr_auc
0,tfidf_linear_svm,0.873893,0.745902,0.707385,0.813506,0.775089
1,PubMedBERT,0.945441,0.912178,0.872340,0.918824,0.937287


Next stage: use the saved PubMedBERT model only after reviewing resource availability and comparing it fairly with the fixed classical test results. No clinical validation is implied.
